In [1]:
!git clone https://github.com/TigistuB21/english-amharic-nmt.git

Cloning into 'english-amharic-nmt'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 39 (delta 1), reused 39 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 6.98 KiB | 3.49 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
%cd /content/english-amharic-nmt
!git status

/content/english-amharic-nmt
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

PROJECT_DATA = Path("/content/drive/MyDrive/english-amharic-nmt-data")

(PROJECT_DATA / "raw").mkdir(parents=True, exist_ok=True)
(PROJECT_DATA / "processed").mkdir(parents=True, exist_ok=True)
(PROJECT_DATA / "models").mkdir(parents=True, exist_ok=True)

print("Created:")
print(PROJECT_DATA)
print()
print("Contents:")
for path in PROJECT_DATA.iterdir():
    print(" -", path.name)

Created:
/content/drive/MyDrive/english-amharic-nmt-data

Contents:
 - raw
 - processed
 - models


In [5]:
!pip install -q datasets

In [6]:
from datasets import load_dataset
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/english-amharic-nmt-data")
RAW_DIR = DATA_DIR / "raw"

dataset = load_dataset(
    "michsethowusu/english-amharic_sentence-pairs_mt560"
)

print(dataset)

README.md:   0%|          | 0.00/861 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  109MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/669145 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['eng', 'amh'],
        num_rows: 669145
    })
})


In [7]:
for i in range(5):
    print("EN:", dataset["train"][i]["eng"])
    print("AM:", dataset["train"][i]["amh"])
    print("-" * 80)

EN: Much of that wisdom concerned Jehovah 's creation : " [ Solomon ] would speak about the trees , from the cedar that is in Lebanon to the hyssop that is coming forth on the wall ; and he would speak about the beasts and about the flying creatures and about the moving things and about the fishes . "
AM: [ ሰሎሞን ] ስለ ዛፍም ከሊባኖስ ዝግባ ጀምሮ በቅጥር ግንብ ላይ እስከሚበቅለው እስከ ሂሶጵ ድረስ ይናገር ነበር ፤ ደግሞም ስለ አውሬዎችና ስለ ወፎች ስለ ተንቀሳቃሾችና ስለ ዓሣዎች ይናገር ነበር ።
--------------------------------------------------------------------------------
EN: A " Necklace to Your Throat "
AM: ለአንገትህም ድሪ ይሆንልሃልና
--------------------------------------------------------------------------------
EN: that He may forgive you some of your sins and respite you until a specified time . Indeed when Allah ' s [ appointed ] time comes , it cannot be deferred , if you know . '
AM: ለእናንተ ከኀጢኣቶቻችሁ ይምራልና ፡ ፡ ወደተወሰነው ጊዜም ያቆያችኋል ፡ ፡ የአላህ ( የወሰነው ) ጊዜ በመጣ ወቅት አይቆይም ፡ ፡ የምታውቁት ብትኾኑ ኖሮ ( በታዘዛችሁ ነበር ) ፡ ፡
-------------------------------------------------

In [8]:
train_data = dataset["train"]

print("Number of sentence pairs:", len(train_data))
print("Columns:", train_data.column_names)

print("\nFirst example:")
print(train_data[0])
import pandas as pd

df = train_data.to_pandas()

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate English sentences:", df["eng"].duplicated().sum())
print("Duplicate Amharic sentences:", df["amh"].duplicated().sum())
print("Duplicate sentence pairs:", df.duplicated(subset=["eng", "amh"]).sum())

df["eng_words"] = df["eng"].str.split().str.len()
df["amh_words"] = df["amh"].str.split().str.len()

print("English word count:")
print(df["eng_words"].describe())

print("\nAmharic word count:")
print(df["amh_words"].describe())

print("Longest English sentences:")
display(
    df.nlargest(5, "eng_words")[["eng", "amh", "eng_words", "amh_words"]]
)

Number of sentence pairs: 669145
Columns: ['eng', 'amh']

First example:
{'eng': 'Much of that wisdom concerned Jehovah \'s creation : " [ Solomon ] would speak about the trees , from the cedar that is in Lebanon to the hyssop that is coming forth on the wall ; and he would speak about the beasts and about the flying creatures and about the moving things and about the fishes . "', 'amh': '[ ሰሎሞን ] ስለ ዛፍም ከሊባኖስ ዝግባ ጀምሮ በቅጥር ግንብ ላይ እስከሚበቅለው እስከ ሂሶጵ ድረስ ይናገር ነበር ፤ ደግሞም ስለ አውሬዎችና ስለ ወፎች ስለ ተንቀሳቃሾችና ስለ ዓሣዎች ይናገር ነበር ።'}
Shape: (669145, 2)

Missing values:
eng    0
amh    0
dtype: int64

Duplicate English sentences: 9062
Duplicate Amharic sentences: 92133
Duplicate sentence pairs: 51
English word count:
count    669145.000000
mean         21.409904
std          12.188779
min           1.000000
25%          13.000000
50%          19.000000
75%          28.000000
max         120.000000
Name: eng_words, dtype: float64

Amharic word count:
count    669145.000000
mean         14.594515
std       

,eng,amh,eng_words,amh_words
181028,That you turn your faces towards the east or t...,መልካም ሥራ ፊቶቻችሁን ወደ ምሥራቅና ምዕራብ አቅጣጫ ማዞር አይደለም ፡ ...,120,88
506566,And We have commanded man to be good towards p...,ሰውንም በወላጆቹ በጎ መዋልን በጥብቅ አዘዝነው ፡ ፡ እናቱ በችግር ላይ ...,120,75
649963,"On March 26 , 1965 , the Student Nonviolent Co...","በመጋቢት 26 , 1965 , ተማሪው Nonviolent Coordinating...",120,100
122274,"To show what makes the days "" hard to deal wit...","አክሎም ይህ ዘመን "" ለመቋቋም የሚያስቸግር "" እንዲሆን የሚያደርገው ምን...",119,94
145342,By Jews I five times received forty strokes le...,ብዙ ጊዜ በመንገድ ሄድሁ ፤ በወንዝ ፍርሃት ፣ በወንበዴዎች ፍርሃት ፣ በ...,119,53


In [9]:
# Exact duplicate sentence pairs
duplicate_pairs = df[df.duplicated(subset=["eng", "amh"], keep=False)]

print("Rows belonging to duplicate pairs:", len(duplicate_pairs))
print("Unique duplicate pairs:", duplicate_pairs[["eng", "amh"]].drop_duplicates().shape[0])

# Length percentiles
print("English length percentiles:")
print(df["eng_words"].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

print("\nAmharic length percentiles:")
print(df["amh_words"].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

thresholds = [30, 40, 50, 60, 80, 100]

print("Sentences exceeding each threshold:\n")

for threshold in thresholds:
    eng_count = (df["eng_words"] > threshold).sum()
    amh_count = (df["amh_words"] > threshold).sum()

    print(
        f">{threshold} words | "
        f"English: {eng_count:,} | "
        f"Amharic: {amh_count:,}"
    )



    display(
    duplicate_pairs[
        ["eng", "amh"]
    ].drop_duplicates().head(10)
)

Rows belonging to duplicate pairs: 100
Unique duplicate pairs: 49
English length percentiles:
0.900    37.0
0.950    44.0
0.990    61.0
0.995    68.0
0.999    88.0
Name: eng_words, dtype: float64

Amharic length percentiles:
0.900    25.0
0.950    30.0
0.990    44.0
0.995    52.0
0.999    72.0
Name: amh_words, dtype: float64
Sentences exceeding each threshold:

>30 words | English: 126,136 | Amharic: 32,116


,eng,amh
1449,"On seeing the crowds he felt pity for them , b...",ብዙ ሕዝብም ባየ ጊዜ ፣ እረኛ እንደ ሌላቸው በጎች ተጨንቀው ተጥለውም ነ...
12965,"Let me devour the food prescribed for me , tha...",ያለዚያ ግን ያለ ልክ እጠግብና እክድሃለሁ ፤ ' እግዚአብሔር ማን ነው ? '
14699,By this the love of God was made manifest in o...,በዚህ የእግዚአብሔር ፍቅር በእኛ ዘንድ ተገለጠ ፣ በእርሱ በኩል በሕይወት...
21683,Awake !,ንቁ !
26976,"As far off as the sunrise is from the sunset ,...",ምሥራቅ ከምዕራብ እንደሚርቅ ፣ መተላለፋችንን በዚያው መጠን ከእኛ አስወገደ ።
30369,"Let your will take place , as in heaven , also...",ፈቃድህ በሰማይ እየሆነ እንዳለ ሁሉ በምድርም ላይ ይሁን ።
49022,I 'm one of Jehovah 's Witnesses .,እኔ የይሖዋ ምሥክር ነኝ ።
53594,Why don 't you salute the flag ?,ብሔራዊ መዝሙር የማትዘምረው ለምንድን ነው ?
63185,Is This Person Right for Me ?,ይህ ሰው ጥሩ የትዳር ጓደኛ ሊሆነኝ ይችላል ?
67352,Let your kingdom come .,መንግሥትህ ይምጣ ።


>40 words | English: 47,346 | Amharic: 9,608


,eng,amh
1449,"On seeing the crowds he felt pity for them , b...",ብዙ ሕዝብም ባየ ጊዜ ፣ እረኛ እንደ ሌላቸው በጎች ተጨንቀው ተጥለውም ነ...
12965,"Let me devour the food prescribed for me , tha...",ያለዚያ ግን ያለ ልክ እጠግብና እክድሃለሁ ፤ ' እግዚአብሔር ማን ነው ? '
14699,By this the love of God was made manifest in o...,በዚህ የእግዚአብሔር ፍቅር በእኛ ዘንድ ተገለጠ ፣ በእርሱ በኩል በሕይወት...
21683,Awake !,ንቁ !
26976,"As far off as the sunrise is from the sunset ,...",ምሥራቅ ከምዕራብ እንደሚርቅ ፣ መተላለፋችንን በዚያው መጠን ከእኛ አስወገደ ።
30369,"Let your will take place , as in heaven , also...",ፈቃድህ በሰማይ እየሆነ እንዳለ ሁሉ በምድርም ላይ ይሁን ።
49022,I 'm one of Jehovah 's Witnesses .,እኔ የይሖዋ ምሥክር ነኝ ።
53594,Why don 't you salute the flag ?,ብሔራዊ መዝሙር የማትዘምረው ለምንድን ነው ?
63185,Is This Person Right for Me ?,ይህ ሰው ጥሩ የትዳር ጓደኛ ሊሆነኝ ይችላል ?
67352,Let your kingdom come .,መንግሥትህ ይምጣ ።


>50 words | English: 17,829 | Amharic: 3,709


...

,eng,amh
1449,"On seeing the crowds he felt pity for them , b...",ብዙ ሕዝብም ባየ ጊዜ ፣ እረኛ እንደ ሌላቸው በጎች ተጨንቀው ተጥለውም ነ...
12965,"Let me devour the food prescribed for me , tha...",ያለዚያ ግን ያለ ልክ እጠግብና እክድሃለሁ ፤ ' እግዚአብሔር ማን ነው ? '
14699,By this the love of God was made manifest in o...,በዚህ የእግዚአብሔር ፍቅር በእኛ ዘንድ ተገለጠ ፣ በእርሱ በኩል በሕይወት...
21683,Awake !,ንቁ !
26976,"As far off as the sunrise is from the sunset ,...",ምሥራቅ ከምዕራብ እንደሚርቅ ፣ መተላለፋችንን በዚያው መጠን ከእኛ አስወገደ ።
30369,"Let your will take place , as in heaven , also...",ፈቃድህ በሰማይ እየሆነ እንዳለ ሁሉ በምድርም ላይ ይሁን ።
49022,I 'm one of Jehovah 's Witnesses .,እኔ የይሖዋ ምሥክር ነኝ ።
53594,Why don 't you salute the flag ?,ብሔራዊ መዝሙር የማትዘምረው ለምንድን ነው ?
63185,Is This Person Right for Me ?,ይህ ሰው ጥሩ የትዳር ጓደኛ ሊሆነኝ ይችላል ?
67352,Let your kingdom come .,መንግሥትህ ይምጣ ።


>80 words | English: 1,148 | Amharic: 339


,eng,amh
1449,"On seeing the crowds he felt pity for them , b...",ብዙ ሕዝብም ባየ ጊዜ ፣ እረኛ እንደ ሌላቸው በጎች ተጨንቀው ተጥለውም ነ...
12965,"Let me devour the food prescribed for me , tha...",ያለዚያ ግን ያለ ልክ እጠግብና እክድሃለሁ ፤ ' እግዚአብሔር ማን ነው ? '
14699,By this the love of God was made manifest in o...,በዚህ የእግዚአብሔር ፍቅር በእኛ ዘንድ ተገለጠ ፣ በእርሱ በኩል በሕይወት...
21683,Awake !,ንቁ !
26976,"As far off as the sunrise is from the sunset ,...",ምሥራቅ ከምዕራብ እንደሚርቅ ፣ መተላለፋችንን በዚያው መጠን ከእኛ አስወገደ ።
30369,"Let your will take place , as in heaven , also...",ፈቃድህ በሰማይ እየሆነ እንዳለ ሁሉ በምድርም ላይ ይሁን ።
49022,I 'm one of Jehovah 's Witnesses .,እኔ የይሖዋ ምሥክር ነኝ ።
53594,Why don 't you salute the flag ?,ብሔራዊ መዝሙር የማትዘምረው ለምንድን ነው ?
63185,Is This Person Right for Me ?,ይህ ሰው ጥሩ የትዳር ጓደኛ ሊሆነኝ ይችላል ?
67352,Let your kingdom come .,መንግሥትህ ይምጣ ።


>100 words | English: 263 | Amharic: 83


,eng,amh
1449,"On seeing the crowds he felt pity for them , b...",ብዙ ሕዝብም ባየ ጊዜ ፣ እረኛ እንደ ሌላቸው በጎች ተጨንቀው ተጥለውም ነ...
12965,"Let me devour the food prescribed for me , tha...",ያለዚያ ግን ያለ ልክ እጠግብና እክድሃለሁ ፤ ' እግዚአብሔር ማን ነው ? '
14699,By this the love of God was made manifest in o...,በዚህ የእግዚአብሔር ፍቅር በእኛ ዘንድ ተገለጠ ፣ በእርሱ በኩል በሕይወት...
21683,Awake !,ንቁ !
26976,"As far off as the sunrise is from the sunset ,...",ምሥራቅ ከምዕራብ እንደሚርቅ ፣ መተላለፋችንን በዚያው መጠን ከእኛ አስወገደ ።
30369,"Let your will take place , as in heaven , also...",ፈቃድህ በሰማይ እየሆነ እንዳለ ሁሉ በምድርም ላይ ይሁን ።
49022,I 'm one of Jehovah 's Witnesses .,እኔ የይሖዋ ምሥክር ነኝ ።
53594,Why don 't you salute the flag ?,ብሔራዊ መዝሙር የማትዘምረው ለምንድን ነው ?
63185,Is This Person Right for Me ?,ይህ ሰው ጥሩ የትዳር ጓደኛ ሊሆነኝ ይችላል ?
67352,Let your kingdom come .,መንግሥትህ ይምጣ ።


In [10]:
import re

clean_df = df[["eng", "amh"]].copy()

# Remove missing values
clean_df = clean_df.dropna(subset=["eng", "amh"])

# Normalize whitespace
clean_df["eng"] = clean_df["eng"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

clean_df["amh"] = clean_df["amh"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

# Remove exact duplicate translation pairs
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["eng", "amh"]
).reset_index(drop=True)

after = len(clean_df)

print("Before cleaning:", before)
print("After removing duplicate pairs:", after)
print("Removed:", before - after)
print("Missing values:")
print(clean_df.isnull().sum())

print("\nDuplicate pairs remaining:")
print(
    clean_df.duplicated(
        subset=["eng", "amh"]
    ).sum()
)

display(clean_df.head(10))

Before cleaning: 669145
After removing duplicate pairs: 669094
Removed: 51
Missing values:
eng    0
amh    0
dtype: int64

Duplicate pairs remaining:
0


,eng,amh
0,Much of that wisdom concerned Jehovah 's creat...,[ ሰሎሞን ] ስለ ዛፍም ከሊባኖስ ዝግባ ጀምሮ በቅጥር ግንብ ላይ እስከሚ...
1,"A "" Necklace to Your Throat """,ለአንገትህም ድሪ ይሆንልሃልና
2,that He may forgive you some of your sins and ...,ለእናንተ ከኀጢኣቶቻችሁ ይምራልና ፡ ፡ ወደተወሰነው ጊዜም ያቆያችኋል ፡ ...
3,Let all malicious bitterness and anger and wra...,መራርነትና ንዴት ቊጣም ጩኸትም መሳደብም ሁሉ ከክፋት ሁሉ ጋር ከእናንተ ...
4,"See the box entitled "" Why Does the Bible Desc...",መጽሐፍ ቅዱስ አምላክን ሰብዓዊ አካል እንዳለው አድርጐ የሚገልጸው ለምንድ...
5,Let Your Reasonableness Become Known to All Men,ምክንያታዊነታችሁ ለሰው ሁሉ ይታወቅ
6,""" So peace is on me the day I was born , the d...",ሰላምም በእኔ ላይ ነው ፡ ፡ በተወለድሁ ቀን ፣ በምሞትበትም ቀን ፣ ሕያ...
7,"As the heavens are higher than the earth , his...",ሰማይ ከምድር ከፍ እንደሚል ፣ እንዲሁ ለሚፈሩት ምሕረቱ ታላቅ ናት ።
8,Will You Be a Giver or a Taker ?,ሰጪ ትሆናላችሁ ወይስ ተቀባዮች ?
9,The Minding of the Spirit,ስለ መንፈስ ማሰብ


In [11]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: divide the remaining 20% equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\nProportions:")
print("Train:", len(train_df) / len(clean_df))
print("Validation:", len(val_df) / len(clean_df))
print("Test:", len(test_df) / len(clean_df))

train_pairs = set(zip(train_df["eng"], train_df["amh"]))
val_pairs = set(zip(val_df["eng"], val_df["amh"]))
test_pairs = set(zip(test_df["eng"], test_df["amh"]))

print("Train ∩ Validation:", len(train_pairs & val_pairs))
print("Train ∩ Test:", len(train_pairs & test_pairs))
print("Validation ∩ Test:", len(val_pairs & test_pairs))

Train: 535275
Validation: 66909
Test: 66910
Total: 669094

Proportions:
Train: 0.7999997010883374
Validation: 0.09999940217667473
Test: 0.10000089673498791
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [12]:
from pathlib import Path

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

print("Saved:")
print(PROCESSED_DIR / "train.csv")
print(PROCESSED_DIR / "validation.csv")
print(PROCESSED_DIR / "test.csv")

import os

for filename in [
    "train.csv",
    "validation.csv",
    "test.csv"
]:
    path = PROCESSED_DIR / filename
    print(filename, "->", os.path.getsize(path) / (1024 * 1024), "MB")


Saved:
/content/drive/MyDrive/english-amharic-nmt-data/processed/train.csv
/content/drive/MyDrive/english-amharic-nmt-data/processed/validation.csv
/content/drive/MyDrive/english-amharic-nmt-data/processed/test.csv
train.csv -> 139.42634105682373 MB
validation.csv -> 17.485118865966797 MB
test.csv -> 17.480867385864258 MB


In [13]:
from collections import Counter

def whitespace_tokenize(text):
    return text.split()

english_counter = Counter()
amharic_counter = Counter()

for text in train_df["eng"]:
    english_counter.update(whitespace_tokenize(text))

for text in train_df["amh"]:
    amharic_counter.update(whitespace_tokenize(text))

print("Unique English tokens:", len(english_counter))
print("Unique Amharic tokens:", len(amharic_counter))

print("\nMost common English tokens:")
print(english_counter.most_common(20))

print("\nMost common Amharic tokens:")
print(amharic_counter.most_common(20))

for threshold in [1, 2, 3, 5, 10]:
    eng_count = sum(
        1 for count in english_counter.values()
        if count >= threshold
    )

    amh_count = sum(
        1 for count in amharic_counter.values()
        if count >= threshold
    )

    print(
        f"Frequency >= {threshold}: "
        f"English={eng_count:,}, "
        f"Amharic={amh_count:,}"
    )

Unique English tokens: 84812
Unique Amharic tokens: 328778

Most common English tokens:
[(',', 602728), ('the', 498901), ('.', 493274), ('of', 302769), ('to', 280421), ('and', 268381), ('"', 206107), ('in', 169104), ('a', 159795), ('that', 132768), ('is', 102779), (':', 102172), ('-', 93979), ('for', 82131), ('you', 77516), ('?', 72343), ("'s", 67151), ('with', 65605), ('be', 65397), ('his', 64208)]

Most common Amharic tokens:
[('።', 385485), ('፡', 170507), ('"', 165615), ('ነው', 90962), ('ላይ', 76384), ('፣', 75693), ('፤', 73544), ('?', 72507), ('-', 56310), (':', 54205), ('ጊዜ', 47806), ('(', 47062), (')', 47034), ("'", 46734), ('ነበር', 45663), ('፥', 44957), ('ውስጥ', 38010), ('ወደ', 37102), ('ሰዎች', 32076), ('ጋር', 31411)]
Frequency >= 1: English=84,812, Amharic=328,778
Frequency >= 2: English=55,308, Amharic=183,421
Frequency >= 3: English=44,100, Amharic=126,990
Frequency >= 5: English=33,959, Amharic=87,401
Frequency >= 10: English=23,772, Amharic=54,395


In [14]:
MIN_FREQ = 5

eng_vocab_tokens = [
    token for token, count in english_counter.items()
    if count >= MIN_FREQ
]

amh_vocab_tokens = [
    token for token, count in amharic_counter.items()
    if count >= MIN_FREQ
]

print("English vocabulary:", len(eng_vocab_tokens))
print("Amharic vocabulary:", len(amh_vocab_tokens))

eng_total = sum(english_counter.values())
amh_total = sum(amharic_counter.values())

eng_unk = sum(
    count for token, count in english_counter.items()
    if count < MIN_FREQ
)

amh_unk = sum(
    count for token, count in amharic_counter.items()
    if count < MIN_FREQ
)

print("English total tokens:", eng_total)
print("English tokens mapped to <UNK>:", eng_unk)
print("English UNK percentage:", eng_unk / eng_total * 100)

print()

print("Amharic total tokens:", amh_total)
print("Amharic tokens mapped to <UNK>:", amh_unk)
print("Amharic UNK percentage:", amh_unk / amh_total * 100)

English vocabulary: 33959
Amharic vocabulary: 87401
English total tokens: 11453705
English tokens mapped to <UNK>: 86387
English UNK percentage: 0.7542275621731134

Amharic total tokens: 7808784
Amharic tokens mapped to <UNK>: 392445
Amharic UNK percentage: 5.025686457712237


In [15]:
from collections import Counter

SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
MIN_FREQ = 5


class Vocabulary:
    def __init__(self, counter, min_freq=5):
        self.special_tokens = SPECIAL_TOKENS

        # Keep tokens that meet the minimum frequency
        tokens = [
            token
            for token, count in counter.items()
            if count >= min_freq
        ]

        # Sort for deterministic vocabulary creation
        tokens = sorted(tokens)

        # Special tokens come first
        self.itos = self.special_tokens + tokens

        # Token -> integer ID
        self.stoi = {
            token: idx
            for idx, token in enumerate(self.itos)
        }

    def __len__(self):
        return len(self.itos)


english_vocab = Vocabulary(
    english_counter,
    min_freq=MIN_FREQ
)

amharic_vocab = Vocabulary(
    amharic_counter,
    min_freq=MIN_FREQ
)

print("English vocabulary size:", len(english_vocab))
print("Amharic vocabulary size:", len(amharic_vocab))

print("\nSpecial token IDs:")
for token in SPECIAL_TOKENS:
    print(
        token,
        "-> English:",
        english_vocab.stoi[token],
        "| Amharic:",
        amharic_vocab.stoi[token]
    )

    test_sentence = train_df.iloc[0]["eng"]

tokens = test_sentence.split()

token_ids = [
    english_vocab.stoi.get(token, english_vocab.stoi["<UNK>"])
    for token in tokens
]

print("Sentence:")
print(test_sentence)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)


test_sentence = train_df.iloc[0]["amh"]

tokens = test_sentence.split()

token_ids = [
    amharic_vocab.stoi.get(token, amharic_vocab.stoi["<UNK>"])
    for token in tokens
]

print("Sentence:")
print(test_sentence)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

English vocabulary size: 33963
Amharic vocabulary size: 87405

Special token IDs:
<PAD> -> English: 0 | Amharic: 0
<UNK> -> English: 1 | Amharic: 1
<SOS> -> English: 2 | Amharic: 2
<EOS> -> English: 3 | Amharic: 3
Sentence:
What is the second woe recorded by Habakkuk , and how can we be sure that dishonest riches will be of no avail ?

Tokens:
['What', 'is', 'the', 'second', 'woe', 'recorded', 'by', 'Habakkuk', ',', 'and', 'how', 'can', 'we', 'be', 'sure', 'that', 'dishonest', 'riches', 'will', 'be', 'of', 'no', 'avail', '?']

Token IDs:
[11320, 22519, 31477, 29040, 33697, 27609, 14256, 4964, 48, 12401, 21371, 14343, 33367, 13327, 30970, 31474, 17348, 28372, 33591, 13327, 25002, 24733, 13054, 830]
Sentence:
ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?

Tokens:
['ዕንባቆም', 'የመዘገበው', 'ሁለተኛው', 'ወዮታ', 'ምንድን', 'ነው', '?']

Token IDs:
[61966, 63814, 918, 60586, 13498, 37703, 623]


In [16]:
def sequence_length(text):
    return len(text.split()) + 2  # + <SOS> and <EOS>


train_df["eng_len"] = train_df["eng"].apply(sequence_length)
train_df["amh_len"] = train_df["amh"].apply(sequence_length)


print("English sequence length:")
print(train_df["eng_len"].describe())

print("\nAmharic sequence length:")
print(train_df["amh_len"].describe())


for lang, column in [
    ("English", "eng_len"),
    ("Amharic", "amh_len")
]:
    print(f"\n{lang}")

    for p in [90, 95, 97, 98, 99, 99.5, 99.9, 100]:
        value = train_df[column].quantile(p / 100)
        print(f"{p}%: {value:.0f}")

        MAX_LENGTHS = [30, 40, 50, 60, 70, 80, 100]

for max_len in MAX_LENGTHS:
    eng_truncated = (train_df["eng_len"] > max_len).sum()
    amh_truncated = (train_df["amh_len"] > max_len).sum()

    print(
        f"MAX_LEN={max_len}: "
        f"English={eng_truncated:,} ({eng_truncated / len(train_df) * 100:.2f}%), "
        f"Amharic={amh_truncated:,} ({amh_truncated / len(train_df) * 100:.2f}%)"
    )

English sequence length:
count    535275.000000
mean         23.397796
std          12.185143
min           3.000000
25%          15.000000
50%          21.000000
75%          30.000000
max         122.000000
Name: eng_len, dtype: float64

Amharic sequence length:
count    535275.000000
mean         16.588359
std           8.618101
min           3.000000
25%          11.000000
50%          15.000000
75%          20.000000
max         120.000000
Name: amh_len, dtype: float64

English
90%: 39
95%: 46
97%: 51
98%: 55
99%: 63
99.5%: 70
99.9%: 90
100%: 122

Amharic
90%: 27
95%: 32
97%: 36
98%: 40
99%: 46
99.5%: 54
99.9%: 74
100%: 120
MAX_LEN=30: English=121,960 (22.78%), Amharic=33,556 (6.27%)
MAX_LEN=40: English=46,087 (8.61%), Amharic=9,673 (1.81%)
MAX_LEN=50: English=17,229 (3.22%), Amharic=3,454 (0.65%)
MAX_LEN=60: English=6,485 (1.21%), Amharic=1,308 (0.24%)
MAX_LEN=70: English=2,521 (0.47%), Amharic=681 (0.13%)
MAX_LEN=80: English=1,052 (0.20%), Amharic=343 (0.06%)
MAX_LEN=100: Englis

In [17]:
!git clone https://github.com/TigistuB21/english-amharic-nmt.git
%cd /content/english-amharic-nmt
import os

print("Current directory:")
print(os.getcwd())

print("\nProject contents:")
print(os.listdir())
!git status

Cloning into 'english-amharic-nmt'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 39 (delta 1), reused 39 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 6.98 KiB | 6.98 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/english-amharic-nmt
Current directory:
/content/english-amharic-nmt

Project contents:
['.gitignore', 'data', 'configs', 'LICENSE', 'requirements.txt', 'app', 'src', 'notebooks', '.git', 'README.md', 'evaluation', 'english-amharic-nmt', 'models']
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	english-amharic-nmt/

nothing added to commit but untracked files present (use "git add" to track)


In [18]:
from pathlib import Path

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/english-amharic-nmt-data")
RAW_DIR = DRIVE_DATA_DIR / "raw"

print("Raw data directory:")
print(RAW_DIR)

print("\nFiles inside raw/:")
for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(path)

Raw data directory:
/content/drive/MyDrive/english-amharic-nmt-data/raw

Files inside raw/:


In [20]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("/content/drive/MyDrive/english-amharic-nmt-data/processed")

train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "validation.csv"
test_path = PROCESSED_DIR / "test.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nFirst training example:")
print("EN:", train_df.iloc[0]["eng"])
print("AM:", train_df.iloc[0]["amh"])

Train shape: (535275, 2)
Validation shape: (66909, 2)
Test shape: (66910, 2)

Columns:
['eng', 'amh']

Missing values:
eng    0
amh    0
dtype: int64

First training example:
EN: What is the second woe recorded by Habakkuk , and how can we be sure that dishonest riches will be of no avail ?
AM: ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?


In [21]:
MAX_LEN = 70

def count_tokens(text):
    return len(str(text).split())

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    eng_lengths = df["eng"].apply(count_tokens)
    amh_lengths = df["amh"].apply(count_tokens)

    # +2 accounts for <SOS> and <EOS>
    keep = (eng_lengths + 2 <= MAX_LEN) & (amh_lengths + 2 <= MAX_LEN)

    print(f"{name}")
    print(f"  Before: {len(df):,}")
    print(f"  Kept:   {keep.sum():,}")
    print(f"  Removed: {(~keep).sum():,}")
    print(f"  Kept %: {keep.mean() * 100:.2f}%")
    print(f"  Removed %: {(~keep).mean() * 100:.2f}%")
    print()

Train
  Before: 535,275
  Kept:   532,412
  Removed: 2,863
  Kept %: 99.47%
  Removed %: 0.53%

Validation
  Before: 66,909
  Kept:   66,545
  Removed: 364
  Kept %: 99.46%
  Removed %: 0.54%

Test
  Before: 66,910
  Kept:   66,533
  Removed: 377
  Kept %: 99.44%
  Removed %: 0.56%



In [22]:
MAX_LEN = 70

def token_length(text):
    return len(str(text).split())

def filter_by_max_length(df, max_len=70):
    eng_len = df["eng"].apply(token_length)
    amh_len = df["amh"].apply(token_length)

    # +2 because every sequence will later receive:
    # <SOS> and <EOS>
    mask = (eng_len + 2 <= max_len) & (amh_len + 2 <= max_len)

    return df.loc[mask].reset_index(drop=True)


# Apply filtering
train_filtered = filter_by_max_length(train_df, MAX_LEN)
val_filtered = filter_by_max_length(val_df, MAX_LEN)
test_filtered = filter_by_max_length(test_df, MAX_LEN)


# Display results
print("Filtered dataset sizes:")
print(f"Train:      {len(train_filtered):,}")
print(f"Validation: {len(val_filtered):,}")
print(f"Test:       {len(test_filtered):,}")
print(f"Total:      {len(train_filtered) + len(val_filtered) + len(test_filtered):,}")

train_filtered.to_csv(
    PROCESSED_DIR / "train_filtered.csv",
    index=False
)

val_filtered.to_csv(
    PROCESSED_DIR / "validation_filtered.csv",
    index=False
)

test_filtered.to_csv(
    PROCESSED_DIR / "test_filtered.csv",
    index=False
)

print("Filtered datasets saved successfully.")

Filtered dataset sizes:
Train:      532,412
Validation: 66,545
Test:       66,533
Total:      665,490
Filtered datasets saved successfully.


In [23]:
for path in PROCESSED_DIR.glob("*filtered.csv"):
    print(path.name, "->", path.stat().st_size / (1024**2), "MB")

train_filtered.csv -> 136.85641384124756 MB
validation_filtered.csv -> 17.15232753753662 MB
test_filtered.csv -> 17.14429759979248 MB


In [24]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed"
)

train_filtered = pd.read_csv(
    PROCESSED_DIR / "train_filtered.csv"
)

val_filtered = pd.read_csv(
    PROCESSED_DIR / "validation_filtered.csv"
)

test_filtered = pd.read_csv(
    PROCESSED_DIR / "test_filtered.csv"
)

print("Train:", train_filtered.shape)
print("Validation:", val_filtered.shape)
print("Test:", test_filtered.shape)

Train: (532412, 2)
Validation: (66545, 2)
Test: (66533, 2)


In [25]:
import pandas as pd
from pathlib import Path
from collections import Counter

# Paths
PROCESSED_DIR = Path(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed"
)

# Load filtered training data
train_filtered = pd.read_csv(
    PROCESSED_DIR / "train_filtered.csv"
)

print("Training dataset:", train_filtered.shape)

# Create token frequency counters
eng_counter = Counter()
amh_counter = Counter()

for sentence in train_filtered["eng"]:
    eng_counter.update(str(sentence).split())

for sentence in train_filtered["amh"]:
    amh_counter.update(str(sentence).split())

print("\nUnique English tokens:", len(eng_counter))
print("Unique Amharic tokens:", len(amh_counter))

Training dataset: (532412, 2)

Unique English tokens: 84251
Unique Amharic tokens: 326667


In [26]:
print("\nMost common English tokens:")
print(eng_counter.most_common(30))

print("\nMost common Amharic tokens:")
print(amh_counter.most_common(30))


Most common English tokens:
[(',', 588464), ('.', 488266), ('the', 488216), ('of', 296158), ('to', 275644), ('and', 260384), ('"', 202937), ('in', 165969), ('a', 157285), ('that', 130238), ('is', 100549), (':', 99555), ('-', 92581), ('for', 80161), ('you', 74152), ('?', 71992), ("'s", 66704), ('with', 64251), ('be', 64111), ('his', 63263), ('I', 62937), ('not', 60294), ('God', 60018), ('was', 56802), ('will', 54345), ('he', 53354), ('it', 52939), ('they', 52122), ('are', 51806), ('Jehovah', 51603)]

Most common Amharic tokens:
[('።', 383877), ('"', 161898), ('፡', 154325), ('ነው', 89334), ('ላይ', 74603), ('፣', 72383), ('?', 72223), ('፤', 71386), ('-', 54882), (':', 52815), ('ጊዜ', 46738), ("'", 46415), ('ነበር', 45329), ('፥', 44295), ('(', 43547), (')', 43519), ('ውስጥ', 37288), ('ወደ', 36124), ('ሰዎች', 31598), ('ጋር', 30890), ('ሁሉ', 30365), ('ነገር', 28444), ('ምን', 24669), ('ሰው', 24034), ('አንድ', 23973), ('.', 23511), ('ይህ', 23391), ('ቅዱስ', 23311), ('እንዲህ', 22134), ('ይሖዋ', 20252)]


In [27]:
thresholds = [1, 2, 3, 5, 10, 20, 50, 100]

print("\nVocabulary size by minimum frequency\n")

print(f"{'Min Frequency':<18} {'English':>12} {'Amharic':>12}")
print("-" * 45)

for threshold in thresholds:
    eng_vocab = sum(
        1 for count in eng_counter.values()
        if count >= threshold
    )

    amh_vocab = sum(
        1 for count in amh_counter.values()
        if count >= threshold
    )

    print(f"{threshold:<18} {eng_vocab:>12,} {amh_vocab:>12,}")


Vocabulary size by minimum frequency

Min Frequency           English      Amharic
---------------------------------------------
1                        84,251      326,667
2                        54,963      181,791
3                        43,820      125,568
5                        33,677       86,123
10                       23,588       52,767
20                       16,420       28,121
50                        9,711       12,783
100                       6,330        6,924


In [28]:
from collections import Counter

# Load the filtered training data
train_df = pd.read_csv(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed/train_filtered.csv"
)

print("Training samples:", len(train_df))

# Tokenize using the same whitespace approach used earlier
eng_counter = Counter()
amh_counter = Counter()

for sentence in train_df["eng"]:
    eng_counter.update(sentence.split())

for sentence in train_df["amh"]:
    amh_counter.update(sentence.split())

print("Unique English tokens:", len(eng_counter))
print("Unique Amharic tokens:", len(amh_counter))

Training samples: 532412
Unique English tokens: 84251
Unique Amharic tokens: 326667


In [29]:
MIN_FREQ = 5

# Build vocabularies using minimum frequency = 5
eng_vocab_words = {
    token for token, count in eng_counter.items()
    if count >= MIN_FREQ
}

amh_vocab_words = {
    token for token, count in amh_counter.items()
    if count >= MIN_FREQ
}

print("English vocabulary:", len(eng_vocab_words))
print("Amharic vocabulary:", len(amh_vocab_words))

English vocabulary: 33677
Amharic vocabulary: 86123


In [30]:
# Count tokens that would become <UNK>
eng_total_tokens = sum(eng_counter.values())
amh_total_tokens = sum(amh_counter.values())

eng_unk_tokens = sum(
    count for token, count in eng_counter.items()
    if count < MIN_FREQ
)

amh_unk_tokens = sum(
    count for token, count in amh_counter.items()
    if count < MIN_FREQ
)

eng_unk_percentage = (eng_unk_tokens / eng_total_tokens) * 100
amh_unk_percentage = (amh_unk_tokens / amh_total_tokens) * 100

print("Vocabulary threshold:", MIN_FREQ)

print("\nEnglish")
print("Vocabulary size:", len(eng_vocab_words))
print("Total tokens:", eng_total_tokens)
print("Tokens mapped to <UNK>:", eng_unk_tokens)
print("UNK percentage:", round(eng_unk_percentage, 4), "%")

print("\nAmharic")
print("Vocabulary size:", len(amh_vocab_words))
print("Total tokens:", amh_total_tokens)
print("Tokens mapped to <UNK>:", amh_unk_tokens)
print("UNK percentage:", round(amh_unk_percentage, 4), "%")

Vocabulary threshold: 5

English
Vocabulary size: 33677
Total tokens: 11233124
Tokens mapped to <UNK>: 86064
UNK percentage: 0.7662 %

Amharic
Vocabulary size: 86123
Total tokens: 7653273
Tokens mapped to <UNK>: 391066
UNK percentage: 5.1098 %


In [31]:
import json
from collections import Counter

MIN_FREQ = 5

# --------------------------------------------------
# 1. Count tokens in the TRAINING DATA ONLY
# --------------------------------------------------

eng_counter = Counter()
amh_counter = Counter()

for sentence in train_df["eng"]:
    eng_counter.update(sentence.split())

for sentence in train_df["amh"]:
    amh_counter.update(sentence.split())


# --------------------------------------------------
# 2. Define special tokens
# --------------------------------------------------

SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<SOS>",
    "<EOS>"
]


# --------------------------------------------------
# 3. Build vocabularies
# --------------------------------------------------

eng_tokens = sorted([
    token for token, count in eng_counter.items()
    if count >= MIN_FREQ
])

amh_tokens = sorted([
    token for token, count in amh_counter.items()
    if count >= MIN_FREQ
])


# --------------------------------------------------
# 4. Special tokens get IDs 0-3
# --------------------------------------------------

eng_vocab = {
    token: idx
    for idx, token in enumerate(SPECIAL_TOKENS + eng_tokens)
}

amh_vocab = {
    token: idx
    for idx, token in enumerate(SPECIAL_TOKENS + amh_tokens)
}


# --------------------------------------------------
# 5. Display results
# --------------------------------------------------

print("English vocabulary size:", len(eng_vocab))
print("Amharic vocabulary size:", len(amh_vocab))

print("\nSpecial tokens:")
for token in SPECIAL_TOKENS:
    print(
        token,
        "-> English:", eng_vocab[token],
        "| Amharic:", amh_vocab[token]
    )

English vocabulary size: 33681
Amharic vocabulary size: 86127

Special tokens:
<PAD> -> English: 0 | Amharic: 0
<UNK> -> English: 1 | Amharic: 1
<SOS> -> English: 2 | Amharic: 2
<EOS> -> English: 3 | Amharic: 3


In [ ]:
import os
import json

VOCAB_DIR = "/content/drive/MyDrive/english-amharic-nmt-data/processed/vocab"

os.makedirs(VOCAB_DIR, exist_ok=True)

with open(f"{VOCAB_DIR}/eng_vocab.json", "w", encoding="utf-8") as f:
    json.dump(eng_vocab, f, ensure_ascii=False, indent=2)

with open(f"{VOCAB_DIR}/amh_vocab.json", "w", encoding="utf-8") as f:
    json.dump(amh_vocab, f, ensure_ascii=False, indent=2)

print("Vocabulary files saved.")

print(os.listdir(VOCAB_DIR))

In [33]:
import json

# Load vocabularies from Google Drive
with open(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed/vocab/eng_vocab.json",
    "r",
    encoding="utf-8"
) as f:
    eng_vocab = json.load(f)

with open(
    "/content/drive/MyDrive/english-amharic-nmt-data/processed/vocab/amh_vocab.json",
    "r",
    encoding="utf-8"
) as f:
    amh_vocab = json.load(f)

# Special token IDs
PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

print("English vocabulary:", len(eng_vocab))
print("Amharic vocabulary:", len(amh_vocab))

def sentence_to_ids(sentence, vocab):
    """
    Convert a sentence into token IDs.

    Example:
        "I am happy"
        ->
        [SOS, I, am, happy, EOS]
    """

    tokens = sentence.split()

    ids = [SOS_IDX]

    for token in tokens:
        token_id = vocab.get(token, UNK_IDX)
        ids.append(token_id)

    ids.append(EOS_IDX)

    return ids


english_sentence = "What is the second woe recorded by Habakkuk , and how can we be sure that dishonest riches will be of no avail ?"

amharic_sentence = "ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?"

eng_ids = sentence_to_ids(
    english_sentence,
    eng_vocab
)

amh_ids = sentence_to_ids(
    amharic_sentence,
    amh_vocab
)

print("English:")
print(eng_ids)

print("\nAmharic:")
print(amh_ids)


unknown_test = "ThisWordDefinitelyDoesNotExistInOurVocabulary"

test_ids = sentence_to_ids(
    unknown_test,
    eng_vocab
)

print(test_ids)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/english-amharic-nmt-data/processed/vocab/eng_vocab.json'

In [34]:
sample = train_df.iloc[0]

print("English:")
print(sample["eng"])

print("\nAmharic:")
print(sample["amh"])

print("\nEnglish IDs:")
print(sentence_to_ids(sample["eng"], eng_vocab))

print("\nAmharic IDs:")
print(sentence_to_ids(sample["amh"], amh_vocab))

English:
What is the second woe recorded by Habakkuk , and how can we be sure that dishonest riches will be of no avail ?

Amharic:
ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?

English IDs:


NameError: name 'sentence_to_ids' is not defined

In [ ]:
MAX_LEN = 70

def pad_sequence_ids(ids, max_len=MAX_LEN):
    """
    Pad or truncate a sequence to max_len.

    Padding uses <PAD> = 0.
    """

    # Truncate if too long
    if len(ids) > max_len:
        ids = ids[:max_len]

        # Make sure the final token remains EOS
        ids[-1] = EOS_IDX

    # Pad if too short
    elif len(ids) < max_len:
        padding = [PAD_IDX] * (max_len - len(ids))
        ids = ids + padding

    return ids


    eng_ids = sentence_to_ids(sample["eng"], eng_vocab)
amh_ids = sentence_to_ids(sample["amh"], amh_vocab)

eng_padded = pad_sequence_ids(eng_ids)
amh_padded = pad_sequence_ids(amh_ids)

print("English length:", len(eng_padded))
print(eng_padded)

print("\nAmharic length:", len(amh_padded))
print(amh_padded)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class TranslationDataset(Dataset):
    def __init__(self, dataframe, src_vocab, trg_vocab, max_len=70):
        self.dataframe = dataframe.reset_index(drop=True)

        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        src_sentence = row["eng"]
        trg_sentence = row["amh"]

        # Sentence → token IDs
        src_ids = sentence_to_ids(
            src_sentence,
            self.src_vocab
        )

        trg_ids = sentence_to_ids(
            trg_sentence,
            self.trg_vocab
        )

        # Padding / truncation
        src_ids = pad_sequence_ids(
            src_ids,
            self.max_len
        )

        trg_ids = pad_sequence_ids(
            trg_ids,
            self.max_len
        )

        # Convert to PyTorch tensors
        src_tensor = torch.tensor(
            src_ids,
            dtype=torch.long
        )

        trg_tensor = torch.tensor(
            trg_ids,
            dtype=torch.long
        )

        return src_tensor, trg_tensor



In [ ]:
train_dataset = TranslationDataset(
    train_df,
    eng_vocab,
    amh_vocab,
    max_len=70
)

print("Dataset size:", len(train_dataset))

In [ ]:
src_tensor, trg_tensor = train_dataset[0]

print("Source tensor:")
print(src_tensor)

print("\nTarget tensor:")
print(trg_tensor)

print("\nSource shape:", src_tensor.shape)
print("Target shape:", trg_tensor.shape)

print("\nData type:", src_tensor.dtype)

In [ ]:
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/english-amharic-nmt-data/processed"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_filtered.csv")
)

validation_df = pd.read_csv(
    os.path.join(DATA_DIR, "validation_filtered.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test_filtered.csv")
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

print("\nMissing values:")
print("Train:")
print(train_df.isnull().sum())

print("Validation:")
print(validation_df.isnull().sum())

print("Test:")
print(test_df.isnull().sum())

In [ ]:
train_dataset = TranslationDataset(
    train_df,
    eng_vocab,
    amh_vocab,
    max_len=70
)

validation_dataset = TranslationDataset(
    validation_df,
    eng_vocab,
    amh_vocab,
    max_len=70
)

test_dataset = TranslationDataset(
    test_df,
    eng_vocab,
    amh_vocab,
    max_len=70
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(validation_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

In [ ]:
source_batch, target_batch = next(iter(train_loader))

print("Source batch:")
print(source_batch)

print("\nTarget batch:")
print(target_batch)

print("\nSource shape:", source_batch.shape)
print("Target shape:", target_batch.shape)

print("\nSource dtype:", source_batch.dtype)
print("Target dtype:", target_batch.dtype)

In [ ]:
PAD_IDX = 0

print("Number of PAD tokens in source batch:",
      (source_batch == PAD_IDX).sum().item())

print("Number of PAD tokens in target batch:",
      (target_batch == PAD_IDX).sum().item())

In [ ]:
# Find the first sample in the batch
source = source_batch[0]
target = target_batch[0]

print("Source:")
print(source.tolist())

print("\nTarget:")
print(target.tolist())

# Find EOS and first PAD positions
source_eos_positions = (source == 3).nonzero(as_tuple=True)[0]
source_pad_positions = (source == 0).nonzero(as_tuple=True)[0]

target_eos_positions = (target == 3).nonzero(as_tuple=True)[0]
target_pad_positions = (target == 0).nonzero(as_tuple=True)[0]

print("\nSource EOS position:",
      source_eos_positions[0].item() if len(source_eos_positions) > 0 else "Not found")

print("Source first PAD position:",
      source_pad_positions[0].item() if len(source_pad_positions) > 0 else "No PAD")

print("\nTarget EOS position:",
      target_eos_positions[0].item() if len(target_eos_positions) > 0 else "Not found")

print("Target first PAD position:",
      target_pad_positions[0].item() if len(target_pad_positions) > 0 else "No PAD")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

In [ ]:
import torch
import torch.nn as nn


INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(amh_vocab)

EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 1
DROPOUT = 0.2


class Encoder(nn.Module):
    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim,
        num_layers=1,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0
        )

    def forward(self, src):
        """
        src shape:
            [source_length, batch_size]
        """

        embedded = self.embedding(src)

        # embedded:
        # [source_length, batch_size, embedding_dim]

        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell




In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(encoder)

In [ ]:
src = source_batch.to(device)

# DataLoader shape:
# [batch_size, sequence_length]

# LSTM expects:
# [sequence_length, batch_size]

src = src.transpose(0, 1)

print("Source shape for encoder:", src.shape)

with torch.no_grad():
    hidden, cell = encoder(src)

print("Hidden shape:", hidden.shape)
print("Cell shape:", cell.shape)

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        hidden_dim,
        num_layers=1,
        dropout=0.2
    ):
        super().__init__()

        self.output_dim = output_dim

        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc_out = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, input, hidden, cell):
        """
        input:
            [batch_size]

        hidden:
            [num_layers, batch_size, hidden_dim]

        cell:
            [num_layers, batch_size, hidden_dim]
        """

        # Add sequence dimension
        input = input.unsqueeze(0)
        # [1, batch_size]

        embedded = self.embedding(input)
        # [1, batch_size, embedding_dim]

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # Remove sequence dimension
        output = output.squeeze(0)
        # [batch_size, hidden_dim]

        prediction = self.fc_out(output)
        # [batch_size, output_dim]

        return prediction, hidden, cell

In [ ]:
decoder = Decoder(
    output_dim=OUTPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(decoder)

In [ ]:
# Take the first target token from the batch.
# This should be <SOS>, whose ID is 2.

trg = target_batch.to(device)

input_token = trg[:, 0]

print("Input token shape:", input_token.shape)
print("First 10 input token IDs:", input_token[:10])

with torch.no_grad():
    prediction, decoder_hidden, decoder_cell = decoder(
        input_token,
        hidden,
        cell
    )

print("Prediction shape:", prediction.shape)
print("Decoder hidden shape:", decoder_hidden.shape)
print("Decoder cell shape:", decoder_cell.shape)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        src:
            [source_length, batch_size]

        trg:
            [target_length, batch_size]

        Returns:
            outputs:
            [target_length, batch_size, output_dim]
        """

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        output_dim = self.decoder.output_dim

        # Store decoder predictions
        outputs = torch.zeros(
            trg_len,
            batch_size,
            output_dim
        ).to(self.device)

        # Encode the source sentence
        hidden, cell = self.encoder(src)

        # First decoder input is <SOS>
        input = trg[0, :]

        # Generate target tokens one at a time
        for t in range(1, trg_len):

            output, hidden, cell = self.decoder(
                input,
                hidden,
                cell
            )

            outputs[t] = output

            # Decide whether to use teacher forcing
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio

            # Best predicted token
            top1 = output.argmax(1)

            # Next decoder input
            input = trg[t] if teacher_force else top1

        return outputs

In [ ]:
model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

print(model)

In [ ]:
# Get one batch
source_batch, target_batch = next(iter(train_loader))

# Move to GPU
source_batch = source_batch.to(device)
target_batch = target_batch.to(device)

# PyTorch Seq2Seq expects:
# [sequence_length, batch_size]
source_batch = source_batch.transpose(0, 1)
target_batch = target_batch.transpose(0, 1)

print("Source shape:", source_batch.shape)
print("Target shape:", target_batch.shape)

# Test complete forward pass
with torch.no_grad():
    output = model(
        source_batch,
        target_batch,
        teacher_forcing_ratio=0.5
    )

print("Output shape:", output.shape)

In [ ]:
import torch.optim as optim

# Ignore <PAD> tokens when calculating loss
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", 0.001)

In [ ]:
def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

total_parameters = count_parameters(model)

print(f"Total trainable parameters: {total_parameters:,}")
print(f"Total trainable parameters: {total_parameters / 1_000_000:.2f} million")

In [ ]:
print("Encoder parameters:")
print(f"{sum(p.numel() for p in encoder.parameters()):,}")

print("\nDecoder parameters:")
print(f"{sum(p.numel() for p in decoder.parameters()):,}")

print("\nOutput layer parameters:")
print(f"{sum(p.numel() for p in decoder.fc_out.parameters()):,}")

In [ ]:
model.train()

# Get one batch
source_batch, target_batch = next(iter(train_loader))

# Move to GPU
source_batch = source_batch.to(device)
target_batch = target_batch.to(device)

# Change from:
# [batch_size, sequence_length]
# to:
# [sequence_length, batch_size]

source_batch = source_batch.transpose(0, 1)
target_batch = target_batch.transpose(0, 1)

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
output = model(
    source_batch,
    target_batch,
    teacher_forcing_ratio=0.5
)

# Remove <SOS> position
output_dim = output.shape[-1]

output = output[1:].reshape(-1, output_dim)
target = target_batch[1:].reshape(-1)

# Calculate loss
loss = criterion(output, target)

print("Loss before backward:", loss.item())

# Backpropagation
loss.backward()

# Update model parameters
optimizer.step()

print("One training step completed.")

In [ ]:
import time
import math

def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()

    epoch_loss = 0

    for source, target in loader:

        source = source.to(device)
        target = target.to(device)

        # [batch, seq_len] → [seq_len, batch]
        source = source.transpose(0, 1)
        target = target.transpose(0, 1)

        optimizer.zero_grad()

        output = model(
            source,
            target,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]

        # Ignore <SOS>
        output = output[1:].reshape(-1, output_dim)
        target = target[1:].reshape(-1)

        loss = criterion(output, target)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            clip
        )

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate_epoch(model, loader, criterion):
    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for source, target in loader:

            source = source.to(device)
            target = target.to(device)

            # [batch, seq_len] → [seq_len, batch]
            source = source.transpose(0, 1)
            target = target.transpose(0, 1)

            output = model(
                source,
                target,
                teacher_forcing_ratio=0
            )

            output_dim = output.shape[-1]

            # Ignore <SOS>
            output = output[1:].reshape(-1, output_dim)
            target = target[1:].reshape(-1)

            loss = criterion(output, target)

            epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [ ]:
model.train()

NUM_TEST_BATCHES = 100

start_time = time.time()

losses = []

for batch_idx, (source, target) in enumerate(train_loader):

    if batch_idx >= NUM_TEST_BATCHES:
        break

    source = source.to(device)
    target = target.to(device)

    # [batch, seq_len] → [seq_len, batch]
    source = source.transpose(0, 1)
    target = target.transpose(0, 1)

    optimizer.zero_grad()

    output = model(
        source,
        target,
        teacher_forcing_ratio=0.5
    )

    output_dim = output.shape[-1]

    output = output[1:].reshape(-1, output_dim)
    target = target[1:].reshape(-1)

    loss = criterion(output, target)

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    losses.append(loss.item())

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Batch {batch_idx + 1}/{NUM_TEST_BATCHES} "
            f"| Loss: {loss.item():.4f}"
        )

elapsed_time = time.time() - start_time

print("\nSanity training completed.")
print(f"Time: {elapsed_time / 60:.2f} minutes")
print(f"First loss: {losses[0]:.4f}")
print(f"Last loss: {losses[-1]:.4f}")

In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/english-amharic-nmt-data/models"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

sanity_checkpoint_path = MODEL_DIR / "seq2seq_sanity_100_batches.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": losses[-1],
    "batch": NUM_TEST_BATCHES,
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "max_len": MAX_LEN,
        "learning_rate": 0.001,
        "teacher_forcing_ratio": 0.5,
    }
}, sanity_checkpoint_path)

print(f"Saved checkpoint to:")
print(sanity_checkpoint_path)

In [ ]:
# Recreate a fresh encoder
encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Recreate a fresh decoder
decoder = Decoder(
    output_dim=OUTPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Recreate the Seq2Seq model
model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

# Fresh optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Official Seq2Seq model initialized from scratch.")
print(f"Parameters: {count_parameters(model):,}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")

In [ ]:
official_untrained_path = MODEL_DIR / "seq2seq_baseline_untrained.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "max_len": MAX_LEN,
        "learning_rate": 0.001,
        "teacher_forcing_ratio": 0.5,
        "parameters": count_parameters(model)
    }
}, official_untrained_path)

print(f"Saved to: {official_untrained_path}")